# Notebook 1 — 2.5D EfficientNet-B4 Training (Session Chaining)
**RSNA Intracranial Hemorrhage Detection — Improved Pipeline**

This notebook handles training across multiple Kaggle sessions.
Each session trains a fixed number of epochs, saves a checkpoint,
and the next session resumes from where the previous one left off.

### Session chaining workflow
| Session | `PREV_CHECKPOINT_DIR` | What happens |
|---------|----------------------|---------------|
| 1 (first run) | `None` (starts fresh) | Trains fold 0 for N epochs |
| 2 | `/kaggle/input/<nb01-session1-name>` | Resumes fold 0 or continues to fold 1 |
| 3 | `/kaggle/input/<nb01-session2-name>` | Continues from where session 2 stopped |

> **Before each session**: update `PREV_CHECKPOINT_DIR` to point to the
> previous session's committed notebook output.

### Key features (matching NB03 + reviewer upgrades)
- **2.5D input**: prev/center/next slices → 9-channel input
- **Multi-label**: 6 outputs (any + 5 subtypes) with weighted BCE
- **EfficientNet-B4**: 19M params, first conv modified for 9ch
- **5-fold GroupKFold CV**: zero patient leakage
- **Hard negative mining**: top-500 FP negatives 3× oversampled
- **MixUp / CutMix**: applied after backbone unfreeze
- **Lock system**: full checkpoint (model + optimizer + scheduler + scaler + patience + early_stopped)
- **NaN divergence guard**: aborts if 5+ NaN losses in one epoch
- **Early stopping**: patience-based on val AUC
- **Discriminative LR**: backbone trains slower than classifier head
- **Backbone freezing**: head-only training for first N epochs

### Inputs
- NB00 output: `manifest_2_5d.csv`
- NB02 (original) output: `cache/`, `normalization_stats.json`
- *(sessions 2+)* Previous session checkpoint

### Epoch plan
- **Per fold**: 20 total epochs (2 frozen warmup + 18 full training)
- **Per session**: 5 epochs (configurable via `N_EPOCHS_THIS_SESSION`)
- **Total**: 5 folds × 20 epochs = 100 epochs max (early stopping reduces this)
- **Sessions needed**: ~3-4 Kaggle sessions (12h each)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  CONFIG — edit these at the start of each session  ██
# ══════════════════════════════════════════════════════════════════════════
from pathlib import Path
import os, json, torch

# ─── SESSION CHAINING (update for sessions 2+) ────────────────────────
PREV_CHECKPOINT_DIR = "/kaggle/input/notebooks/harshitghosh/01-training"   # Session 1: None (fresh start)
# PREV_CHECKPOINT_DIR = '/kaggle/input/nb01-session-1/'  # Session 2+

N_EPOCHS_THIS_SESSION = 5    # Epochs per session (same as NB03)
TOTAL_EPOCHS          = 20   # Total epochs per fold

# ─── Hyperparameters ──────────────────────────────────────────────────
BACKBONE     = 'tf_efficientnet_b4'
IMG_SIZE     = 256
IN_CHANNELS  = 9          # 3 windows × 3 slices (2.5D)
N_CLASSES    = 6          # any + 5 subtypes
BATCH_SIZE   = 16         # GPU batch size
ACCUM_STEPS  = 4          # Effective batch = 16 × 4 = 64
BASE_LR      = 3e-4       # Main learning rate
WARMUP_LR    = 1e-3       # LR for classifier during warmup
BACKBONE_LR_FACTOR = 0.1  # Backbone LR = BASE_LR × this (discriminative)
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 4
SEED         = 42

# ─── Anti-overfitting controls (same spirit as NB03) ─────────────────
FREEZE_BACKBONE_EPOCHS = 2   # Head-only training for first N epochs
PATIENCE               = 5   # Early stopping: epochs without AUC improvement
DROPOUT                = 0.3  # Classifier head dropout
GRAD_CLIP              = 1.0
LABEL_SMOOTH           = 0.05
ANY_WEIGHT             = 1.0  # Loss weight for 'any'
SUB_WEIGHT             = 0.4  # Loss weight for subtypes
MIXUP_ALPHA            = 0.4
CUTMIX_ALPHA           = 1.0

# ─── Hard negative mining ────────────────────────────────────────────
MINE_EVERY   = 5          # Refresh interval (epochs)
MINE_TOP_K   = 500        # Number of hard negatives to boost
MINE_BOOST   = 3.0        # Sampling weight multiplier

N_FOLDS = 5

SUBTYPES = ['any', 'epidural', 'intraparenchymal',
            'intraventricular', 'subarachnoid', 'subdural']

# ─── Paths ────────────────────────────────────────────────────────────
NB00_DIR = Path('/kaggle/input/notebooks/harshitghosh/00-preprocess-metadata')
NB02_DIR = Path('/kaggle/input/notebooks/harshitghosh/nb02eda')

MANIFEST_PATH   = NB00_DIR / 'manifest_2_5d.csv'
NPY_CACHE_DIR   = NB02_DIR / 'cache'
NORM_STATS_PATH = NB02_DIR / 'normalization_stats.json'

CHECKPOINT_PATH    = Path('/kaggle/working/checkpoint.pth')   # current fold
BEST_MODEL_DIR     = Path('/kaggle/working')                  # best_model_fold{k}.pth
METRICS_LOG_PATH   = Path('/kaggle/working/training_metrics.csv')
SESSION_STATE_PATH = Path('/kaggle/working/session_state.json')

# ═══════════════════════════════════════════════════════════════════════
# ██  ENVIRONMENT INFO  ██
# ═══════════════════════════════════════════════════════════════════════
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ═══════════════════════════════════════════════════════════════════════
# ██  VERIFY INPUT DATASETS  ██
# ═══════════════════════════════════════════════════════════════════════
print('\nChecking mounted input datasets...')
print('Available input folders:')
print(os.listdir('/kaggle/input'))

assert NB00_DIR.exists(),     f'❌ NB00 output not found: {NB00_DIR}'
assert MANIFEST_PATH.exists(),f'❌ manifest_2_5d.csv not found: {MANIFEST_PATH}'
assert NPY_CACHE_DIR.exists(),f'❌ NPY cache not found: {NPY_CACHE_DIR}'

npy_count = len(list(NPY_CACHE_DIR.glob('*.npy')))
print(f'\n✅ NB00 manifest found: {MANIFEST_PATH}')
print(f'✅ NB02 cache found: {npy_count:,} NPY files')

# Normalization stats
if NORM_STATS_PATH.exists():
    with open(NORM_STATS_PATH) as f:
        _ns = json.load(f)
    print(f'✅ Normalization stats: mean={_ns["mean"]}, std={_ns["std"]}')
else:
    print('⚠ normalization_stats.json not found, will use ImageNet defaults')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  IMPORTS + SEED  ██
# ══════════════════════════════════════════════════════════════════════════
import gc, time, random, shutil, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


def seed_everything(s: int):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
print('Imports OK.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  DATASET: 2.5D  ██
# ══════════════════════════════════════════════════════════════════════════

# Load normalization stats
# ──────────────────────────────────────────────────────────────────────
# REVIEWER NOTE — Normalization leakage acknowledgment:
# These channel-wise mean/std statistics were computed on the ENTIRE NB02
# cache (all images, not train-only). This constitutes minor statistical
# leakage of first-order pixel moments. Impact on model decision boundaries
# is negligible (pixel-level mean/std are stable across random splits of
# medical images), but we acknowledge this as a limitation. A stricter
# approach would compute stats per-fold on training images only.
# ──────────────────────────────────────────────────────────────────────
if NORM_STATS_PATH.exists():
    with open(NORM_STATS_PATH) as f:
        _ns = json.load(f)
    MEAN_3CH = np.array(_ns['mean'], dtype=np.float32)
    STD_3CH  = np.array(_ns['std'],  dtype=np.float32)
else:
    MEAN_3CH = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    STD_3CH  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Tile to 9 channels: [brain, subdural, soft] × [prev, center, next]
MEAN_9CH = np.tile(MEAN_3CH, 3)
STD_9CH  = np.tile(STD_3CH, 3)


class ICH2_5DDataset(Dataset):
    """Loads 3 adjacent slices (prev, center, next) → [H, W, 9] for 2.5D.
    Reuses existing NB02 NPY cache — no reprocessing needed.
    """

    def __init__(self, df, npy_root, transform, label_col=None):
        self.df = df.reset_index(drop=True)
        self.npy_root = Path(npy_root)
        self.transform = transform
        self.label_col = label_col  # None = multi-label (6 cols)

    def __len__(self):
        return len(self.df)

    def _load_npy(self, image_id):
        if pd.isna(image_id):
            return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        p = self.npy_root / f'{image_id}.npy'
        if p.exists():
            try:
                return np.load(str(p)).astype(np.uint8)
            except Exception:
                pass
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prev_img   = self._load_npy(row.get('prev_image_id'))
        center_img = self._load_npy(row['image_id'])
        next_img   = self._load_npy(row.get('next_image_id'))

        img_9ch = np.concatenate([prev_img, center_img, next_img], axis=2)

        if self.transform:
            img_9ch = self.transform(image=img_9ch)['image']

        labels = torch.tensor([row[c] for c in SUBTYPES], dtype=torch.float32)
        return img_9ch, labels


# ─── Transforms ───────────────────────────────────────────────────────
train_transform = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0),
                        ratio=(0.9, 1.1), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                       rotate_limit=15, p=0.5, border_mode=0),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
                    fill_value=0, p=0.3),
    A.Normalize(mean=MEAN_9CH.tolist(), std=STD_9CH.tolist(),
                max_pixel_value=255.0),
    ToTensorV2(),

])

print('Dataset class + transforms defined.')



val_transform = A.Compose([
    A.Normalize(
        mean=MEAN_9CH.tolist(),
        std=STD_9CH.tolist(),
        max_pixel_value=255.0
    ),
    ToTensorV2(),
])

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  MODEL: 2.5D EfficientNet-B4  ██
# ══════════════════════════════════════════════════════════════════════════

def build_model(backbone=BACKBONE, in_ch=IN_CHANNELS, n_cls=N_CLASSES,
                pretrained=True):
    """Build 2.5D EfficientNet-B4 with modified first conv."""
    model = timm.create_model(backbone, pretrained=pretrained,
                              num_classes=0, drop_rate=DROPOUT,
                              drop_path_rate=0.2)

    # Modify first conv: 3ch → 9ch
    old_conv = model.conv_stem
    new_conv = nn.Conv2d(
        in_ch, old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=(old_conv.bias is not None)
    )
    # Init: repeat pretrained 3ch weights 3 times, scale by 1/3
    k = in_ch // 3
    with torch.no_grad():
        new_conv.weight.copy_(old_conv.weight.repeat(1, k, 1, 1) / k)
        if old_conv.bias is not None:
            new_conv.bias.copy_(old_conv.bias)
    model.conv_stem = new_conv

    # Classifier head
    n_feat = model.num_features  # 1792 for B4
    model.classifier = nn.Sequential(
        nn.Dropout(p=DROPOUT),
        nn.Linear(n_feat, n_cls)
    )

    return model


def set_backbone_frozen(model, freeze: bool):
    """Freeze or unfreeze all backbone layers (keep classifier + conv_stem trainable)."""
    for name, p in model.named_parameters():
        if 'classifier' in name or 'conv_stem' in name:
            p.requires_grad = True
        else:
            p.requires_grad = not freeze
    tag = 'FROZEN' if freeze else 'UNFROZEN'
    n_locked = sum(1 for p in model.parameters() if not p.requires_grad)
    print(f'  Backbone {tag} — {n_locked} parameter tensors locked')


# Quick test
_m = build_model(pretrained=False)
n_params = sum(p.numel() for p in _m.parameters()) / 1e6
print(f'{BACKBONE} 2.5D: {n_params:.1f}M params')
print(f'  conv_stem: {_m.conv_stem.in_channels}ch → {_m.conv_stem.out_channels}ch')
print(f'  backbone features: {_m.num_features}')
with torch.no_grad():
    _out = _m(torch.randn(2, IN_CHANNELS, IMG_SIZE, IMG_SIZE))
    print(f'  forward: [2, {IN_CHANNELS}, {IMG_SIZE}, {IMG_SIZE}] → {list(_out.shape)}')
del _m

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  LOSS + MIXUP / CUTMIX + HARD NEGATIVE MINING  ██
# ══════════════════════════════════════════════════════════════════════════

class MultiLabelLoss(nn.Module):
    """Weighted BCE with label smoothing. 'any' gets higher weight."""
    def __init__(self, any_w=ANY_WEIGHT, sub_w=SUB_WEIGHT, ls=LABEL_SMOOTH):
        super().__init__()
        self.any_w = any_w
        self.sub_w = sub_w
        self.ls = ls

    def forward(self, logits, targets):
        t = targets * (1 - self.ls) + 0.5 * self.ls  # label smoothing
        bce = F.binary_cross_entropy_with_logits(logits, t, reduction='none')
        w = torch.ones_like(bce)
        w[:, 0] = self.any_w   # 'any' column
        w[:, 1:] = self.sub_w  # subtype columns
        return (bce * w).mean()


def rand_bbox(size, lam):
    H, W = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_h, cut_w = int(H * cut_rat), int(W * cut_rat)
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    return y1, x1, y2, x2


def apply_mix(images, targets, epoch, p=0.5):
    """Apply MixUp or CutMix (only after warmup). Returns mixed_images, ta, tb, lam."""
    if epoch < FREEZE_BACKBONE_EPOCHS or random.random() > p:
        return images, targets, targets, 1.0
    idx = torch.randperm(images.size(0), device=images.device)
    if random.random() < 0.5:  # MixUp
        lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
        images = lam * images + (1 - lam) * images[idx]
    else:  # CutMix
        lam = np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA)
        y1, x1, y2, x2 = rand_bbox(images.size(), lam)
        images[:, :, y1:y2, x1:x2] = images[idx, :, y1:y2, x1:x2]
        lam = 1 - (y2-y1)*(x2-x1) / (images.size(-2)*images.size(-1))
    return images, targets, targets[idx], lam


@torch.no_grad()
def mine_hard_negatives(model, dataset, device, top_k=MINE_TOP_K):
    """Forward pass → find top-K false-positive negatives.

    REVIEWER NOTE: This function is ALWAYS called with `train_ds` (see main
    loop below). It never sees validation data, so there is no information
    leakage from hard negative mining into evaluation.
    """
    model.eval()
    loader = DataLoader(dataset, batch_size=BATCH_SIZE*2, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    all_probs, all_labels = [], []
    for imgs, labels in tqdm(loader, desc='  Mining', leave=False):
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
        all_probs.append(torch.sigmoid(logits[:, 0]).cpu())
        all_labels.append(labels[:, 0])
    probs  = torch.cat(all_probs)
    labels = torch.cat(all_labels)
    neg_mask  = labels < 0.5
    neg_probs = probs[neg_mask]
    neg_idxs  = torch.arange(len(probs))[neg_mask]
    k = min(top_k, len(neg_probs))
    _, top_pos = neg_probs.topk(k)
    hard_set = set(neg_idxs[top_pos].tolist())
    print(f'  Mined {len(hard_set)} hard negatives '
          f'(max FP prob={neg_probs[top_pos[0]]:.3f})')
    return hard_set


def build_sampler(n_samples, hard_neg_indices=None):
    weights = np.ones(n_samples, dtype=np.float64)
    if hard_neg_indices:
        for idx in hard_neg_indices:
            if 0 <= idx < n_samples:
                weights[idx] *= MINE_BOOST
    return WeightedRandomSampler(weights, num_samples=n_samples, replacement=True)


print('Loss / MixUp / CutMix / hard-neg mining defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  LOAD MANIFEST + SESSION STATE  ██
# ══════════════════════════════════════════════════════════════════════════
manifest = pd.read_csv(MANIFEST_PATH)
print(f'Manifest: {len(manifest):,} images, {manifest["patient_id"].nunique():,} patients')
print(f'Folds  : {sorted(manifest["fold"].unique())}')

# ─── Restore session state ────────────────────────────────────────────
START_FOLD       = 0
START_EPOCH      = 0
completed_folds  = []
fold_results     = {}           # {fold: best_auc}
metrics_history  = []
best_val_auc     = 0.0
patience_counter = 0
early_stopped    = False

if PREV_CHECKPOINT_DIR is not None:
    prev_dir = Path(PREV_CHECKPOINT_DIR)
    state_path = prev_dir / 'session_state.json'

    if state_path.exists():
        with open(state_path) as f:
            prev_state = json.load(f)

        completed_folds = prev_state.get('completed_folds', [])
        fold_results    = {int(k): v for k, v in prev_state.get('fold_aucs', {}).items()}
        START_FOLD      = prev_state.get('current_fold', 0)
        START_EPOCH     = prev_state.get('current_fold_epoch', 0)

        print(f'\nResuming from previous session:')
        print(f'  Completed folds: {completed_folds}')
        print(f'  Resuming fold {START_FOLD}, epoch {START_EPOCH}')

        # Copy completed fold models
        for k in completed_folds:
            src = prev_dir / f'best_model_fold{k}.pth'
            dst = BEST_MODEL_DIR / f'best_model_fold{k}.pth'
            if src.exists() and not dst.exists():
                shutil.copy2(src, dst)
                print(f'  Copied best_model_fold{k}.pth')

        # Copy metrics log
        prev_metrics = prev_dir / 'training_metrics.csv'
        if prev_metrics.exists():
            metrics_history = pd.read_csv(prev_metrics).to_dict('records')
            print(f'  Loaded {len(metrics_history)} previous epoch records')

    else:
        print(f'⚠ session_state.json not found in {prev_dir}')
        print('  Starting from scratch.')

    # Load fold checkpoint (optimizer/scheduler state)
    prev_ckpt = prev_dir / 'checkpoint.pth'
    if prev_ckpt.exists() and START_EPOCH > 0:
        shutil.copy2(prev_ckpt, CHECKPOINT_PATH)
        print(f'  Copied checkpoint.pth for fold {START_FOLD}')
        
        # Also copy best_model for the current fold if it exists (mid-fold resume)
        src_best = prev_dir / f'best_model_fold{START_FOLD}.pth'
        dst_best = BEST_MODEL_DIR / f'best_model_fold{START_FOLD}.pth'
        if src_best.exists() and not dst_best.exists():
            shutil.copy2(src_best, dst_best)
            print(f'  Copied best_model_fold{START_FOLD}.pth (mid-fold resume)')
else:
    print('No previous session. Starting from scratch (fold 0, epoch 0).')

print(f'\nWill train: fold {START_FOLD} ep{START_EPOCH} → ...'
      f' (up to fold {N_FOLDS-1}, ep{TOTAL_EPOCHS-1})')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  EVALUATION FUNCTION (same spirit as NB03)  ██
# ══════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Run inference on loader and return metrics dict."""
    model.eval()
    all_logits, all_labels = [], []
    running_loss = 0.0
    n_batches = 0

    for imgs, labels in tqdm(loader, desc='  Val', leave=False):
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=False):
            logits = model(imgs)
            loss = criterion(logits, labels)
        running_loss += loss.item()
        n_batches += 1
        all_logits.append(logits.cpu().float())
        all_labels.append(labels.cpu())

    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()

    if not np.isfinite(logits_np).all():
        bad = np.size(logits_np) - np.isfinite(logits_np).sum()
        print(f'  ⚠ Non-finite logits in val: {bad} values; replacing with 0')
        logits_np = np.nan_to_num(logits_np, nan=0.0, posinf=0.0, neginf=0.0)

    probs_np  = 1 / (1 + np.exp(-logits_np))

    if not np.isfinite(probs_np).all():
        bad = np.size(probs_np) - np.isfinite(probs_np).sum()
        print(f'  ⚠ Non-finite probs in val: {bad} values; replacing with 0/1 bounds')
        probs_np = np.nan_to_num(probs_np, nan=0.0, posinf=1.0, neginf=0.0)

    val_loss = running_loss / max(n_batches, 1)

    # Per-class AUC
    aucs = {}
    for i, name in enumerate(SUBTYPES):
        try:
            aucs[name] = roc_auc_score(labels_np[:, i], probs_np[:, i])
        except ValueError:
            aucs[name] = 0.5
    aucs['mean'] = np.mean(list(aucs.values()))

    # Sensitivity / Specificity for 'any' at Youden-optimal threshold
    # REVIEWER NOTE: threshold is computed exclusively from the VALIDATION
    # fold's ROC curve. It never sees training data or other folds.
    fpr, tpr, thresholds = roc_curve(labels_np[:, 0], probs_np[:, 0])
    youden_idx = np.argmax(tpr - fpr)
    best_thresh = float(thresholds[youden_idx])
    preds = (probs_np[:, 0] >= best_thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels_np[:, 0], preds).ravel()
    sensitivity  = tp / (tp + fn + 1e-9)
    specificity  = tn / (tn + fp + 1e-9)
    precision    = tp / (tp + fp + 1e-9)
    f1           = 2 * precision * sensitivity / (precision + sensitivity + 1e-9)

    return {
        'val_loss'    : round(val_loss, 5),
        'auc_any'     : round(aucs['any'], 5),
        'auc_mean'    : round(aucs['mean'], 5),
        'sensitivity' : round(float(sensitivity), 5),
        'specificity' : round(float(specificity), 5),
        'precision'   : round(float(precision), 5),
        'f1'          : round(float(f1), 5),
        'best_thresh' : round(best_thresh, 4),
        'aucs'        : {k: round(v, 5) for k, v in aucs.items()},
    }

print('Evaluation function defined.')

### Checkpoint loading note
PyTorch 2.6+ defaults `torch.load(..., weights_only=True)`, which can fail on checkpoints that include optimizer or scheduler state.
This notebook explicitly uses `weights_only=False` for trusted self-generated checkpoints.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  MAIN TRAINING LOOP (5-fold, session-aware, with locks)  ██
# ══════════════════════════════════════════════════════════════════════════

session_start = time.time()
session_done  = False

for fold_id in range(START_FOLD, N_FOLDS):
    if session_done:
        break

    print(f'\n{"="*65}')
    print(f'  FOLD {fold_id}/{N_FOLDS-1}')
    print(f'{"="*65}')
    fold_start = time.time()

    # ── Split ──────────────────────────────────────────────────────────
    train_df = manifest[manifest['fold'] != fold_id]
    val_df   = manifest[manifest['fold'] == fold_id]
    print(f'  Train: {len(train_df):,}  Val: {len(val_df):,}')

    # ── Datasets ───────────────────────────────────────────────────────
    train_ds = ICH2_5DDataset(train_df, NPY_CACHE_DIR, train_transform)
    val_ds   = ICH2_5DDataset(val_df,   NPY_CACHE_DIR, val_transform)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

    # ── Model ──────────────────────────────────────────────────────────
    model = build_model(pretrained=True).to(DEVICE)
    criterion = MultiLabelLoss()
    scaler = torch.amp.GradScaler('cuda')

    # ── Determine epoch range for this fold in this session ────────────
    ep_start = START_EPOCH if fold_id == START_FOLD else 0
    ep_budget = N_EPOCHS_THIS_SESSION  # how many epochs left in this session

    # Reset fold tracking
    best_val_auc     = 0.0
    patience_counter = 0
    early_stopped    = False
    hard_neg_set     = None

    # ── Load fold checkpoint if resuming mid-fold ──────────────────────
    if ep_start > 0 and CHECKPOINT_PATH.exists():
        print(f'  Loading checkpoint for fold {fold_id}, epoch {ep_start}...')
        ckpt = torch.load(str(CHECKPOINT_PATH), map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        best_val_auc     = ckpt.get('best_val_auc', 0.0)
        patience_counter = ckpt.get('patience_counter', 0)
        early_stopped    = ckpt.get('early_stopped', False)

        if early_stopped:
            print(f'  ⚠ This fold already early-stopped. Skipping to next fold.')
            completed_folds.append(fold_id)
            fold_results[fold_id] = best_val_auc
            ep_start = 0
            START_EPOCH = 0
            continue

        print(f'  Resumed: best_auc={best_val_auc:.5f}, '
              f'patience={patience_counter}/{PATIENCE}')

    # ── Setup optimizers (discriminative LR like NB03) ─────────────────
    # (Will be rebuilt at each phase transition)
    optimizer = None
    scheduler = None

    # Restore optimizer state if resuming
    if ep_start > 0 and CHECKPOINT_PATH.exists():
        try:
            # We need matching optimizer before loading state
            if ep_start < FREEZE_BACKBONE_EPOCHS:
                set_backbone_frozen(model, freeze=True)
                trainable = [p for p in model.parameters() if p.requires_grad]
                optimizer = optim.Adam(trainable, lr=WARMUP_LR)
            else:
                set_backbone_frozen(model, freeze=False)
                backbone_p = [p for n, p in model.named_parameters()
                              if 'classifier' not in n]
                head_p     = [p for n, p in model.named_parameters()
                              if 'classifier' in n]
                optimizer = optim.AdamW([
                    {'params': backbone_p, 'lr': BASE_LR * BACKBONE_LR_FACTOR},
                    {'params': head_p,     'lr': BASE_LR},
                ], weight_decay=WEIGHT_DECAY)

            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)

            saved_groups = len(ckpt['optimizer_state_dict']['param_groups'])
            if saved_groups == len(optimizer.param_groups):
                optimizer.load_state_dict(ckpt['optimizer_state_dict'])
                scheduler.load_state_dict(ckpt['scheduler_state_dict'])
                scaler.load_state_dict(ckpt['scaler_state_dict'])
            else:
                print(f'  ⚠ Optimizer layout changed — weights loaded, optimizer reset')
        except Exception as e:
            print(f'  ⚠ Could not restore optimizer: {e}')
            optimizer = None  # Will be rebuilt below

    for epoch in range(ep_start, TOTAL_EPOCHS):
        ep_time_start = time.time()

        # ── Check session time budget ──────────────────────────────────
        elapsed_h = (time.time() - session_start) / 3600
        if elapsed_h > 10.5:
            print(f'\n  ⏰ Approaching 12h Kaggle limit ({elapsed_h:.1f}h). Stopping.')
            session_done = True
            break

        # Check session epoch budget
        if ep_budget <= 0:
            print(f'  Session epoch budget exhausted. Stopping.')
            session_done = True
            break

        # ── Phase control: freeze/unfreeze backbone ────────────────────
        if epoch == 0 or (epoch == ep_start and ep_start < FREEZE_BACKBONE_EPOCHS):
            set_backbone_frozen(model, freeze=True)
            trainable = [p for p in model.parameters() if p.requires_grad]
            optimizer = optim.Adam(trainable, lr=WARMUP_LR)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)
            phase_name = 'warmup [head-only]'

        if epoch == FREEZE_BACKBONE_EPOCHS:
            set_backbone_frozen(model, freeze=False)
            backbone_p = [p for n, p in model.named_parameters()
                          if 'classifier' not in n]
            head_p     = [p for n, p in model.named_parameters()
                          if 'classifier' in n]
            optimizer = optim.AdamW([
                {'params': backbone_p, 'lr': BASE_LR * BACKBONE_LR_FACTOR},
                {'params': head_p,     'lr': BASE_LR},
            ], weight_decay=WEIGHT_DECAY)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=TOTAL_EPOCHS - FREEZE_BACKBONE_EPOCHS, eta_min=1e-6)
            print(f'  🔓 Backbone unfrozen at epoch {epoch}')
            phase_name = 'main (just unfrozen)'
        elif epoch > FREEZE_BACKBONE_EPOCHS:
            phase_name = 'main'

        # ── Hard negative mining refresh ───────────────────────────────
        if (epoch >= FREEZE_BACKBONE_EPOCHS and
            epoch % MINE_EVERY == 0 and epoch > 0):
            # REVIEWER NOTE: HNM uses TRAINING data only — no val leakage
            hard_neg_set = mine_hard_negatives(model, train_ds, DEVICE)

        # ── Build train loader ─────────────────────────────────────────
        sampler = build_sampler(len(train_ds), hard_neg_set)
        train_loader = DataLoader(
            train_ds, batch_size=BATCH_SIZE, sampler=sampler,
            num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

        # ── TRAIN ──────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        n_batches  = 0
        nan_count  = 0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(train_loader,
                    desc=f'  Ep{epoch:02d} [{phase_name}]', leave=True)
        for step, (imgs, labels) in enumerate(pbar):
            imgs   = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            # MixUp / CutMix
            imgs, ta, tb, lam = apply_mix(imgs, labels, epoch)

            with torch.amp.autocast('cuda'):
                logits = model(imgs)
                loss = (lam * criterion(logits, ta)
                        + (1 - lam) * criterion(logits, tb))
                loss = loss / ACCUM_STEPS

            # NaN / Inf divergence guard
            if not torch.isfinite(loss):
                nan_count += 1
                if nan_count >= 5:
                    raise RuntimeError(
                        f'Training diverged: {nan_count} NaN/Inf losses in epoch {epoch}. '
                        f'Try reducing lr or batch size.')
                print(f'  ⚠ NaN/Inf loss at step {step} — skipping')
                optimizer.zero_grad(set_to_none=True)
                continue

            scaler.scale(loss).backward()

            if (step + 1) % ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            train_loss += loss.item() * ACCUM_STEPS
            n_batches  += 1
            pbar.set_postfix(loss=f'{train_loss/n_batches:.4f}')

        scheduler.step()

        if n_batches == 0:
            raise RuntimeError(f'No valid batches in epoch {epoch}!')

        train_loss /= n_batches

        # ── VALIDATE ───────────────────────────────────────────────────
        metrics = evaluate(model, val_loader, criterion, DEVICE)

        ep_elapsed = time.time() - ep_time_start
        lrs = scheduler.get_last_lr()

        row = {
            'fold': fold_id, 'epoch': epoch, 'phase': phase_name,
            'train_loss': round(train_loss, 5),
            'val_loss': metrics['val_loss'],
            'auc_any': metrics['auc_any'],
            'auc_mean': metrics['auc_mean'],
            'sensitivity': metrics['sensitivity'],
            'specificity': metrics['specificity'],
            'f1': metrics['f1'],
            'best_thresh': metrics['best_thresh'],
            'lr': lrs[0],
            'nan_batches': nan_count,
            'elapsed_s': round(ep_elapsed, 1),
        }
        metrics_history.append(row)

        # Save metrics log (always)
        pd.DataFrame(metrics_history).to_csv(METRICS_LOG_PATH, index=False)

        # ── Early stopping check ───────────────────────────────────────
        if metrics['auc_mean'] > best_val_auc:
            best_val_auc = metrics['auc_mean']
            patience_counter = 0
            # Save best model
            torch.save(model.state_dict(),
                       BEST_MODEL_DIR / f'best_model_fold{fold_id}.pth')
            print(f'  ★ New best AUC_mean={best_val_auc:.5f} — saved')
        else:
            patience_counter += 1

        # ── Save checkpoint (always, every epoch) ──────────────────────
        torch.save({
            'fold': fold_id, 'epoch': epoch,
            'backbone': BACKBONE,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_val_auc': best_val_auc,
            'patience_counter': patience_counter,
            'early_stopped': False,
            'best_thresh': metrics['best_thresh'],
        }, CHECKPOINT_PATH)

        print(f'  Ep{epoch:02d} [{phase_name:20s}] '
              f'trn={train_loss:.4f} val={metrics["val_loss"]:.4f} '
              f'AUC_any={metrics["auc_any"]:.4f} AUC_mean={metrics["auc_mean"]:.4f} '
              f'Sens={metrics["sensitivity"]:.4f} Spec={metrics["specificity"]:.4f} '
              f'pat={patience_counter}/{PATIENCE} NaN={nan_count} '
              f'{ep_elapsed:.0f}s')

        ep_budget -= 1

        # ── Early stopping break ───────────────────────────────────────
        if patience_counter >= PATIENCE and epoch >= FREEZE_BACKBONE_EPOCHS:
            print(f'\n  ⛔ Early stopping fold {fold_id} at epoch {epoch}')
            early_stopped = True
            ckpt_es = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
            ckpt_es['early_stopped'] = True
            torch.save(ckpt_es, CHECKPOINT_PATH)
            break

    # ── Fold complete ──────────────────────────────────────────────────
    if not session_done:
        completed_folds.append(fold_id)
        fold_results[fold_id] = best_val_auc
        fold_time = (time.time() - fold_start) / 3600
        print(f'\n  Fold {fold_id} done — best AUC_mean={best_val_auc:.5f} — '
              f'{fold_time:.1f}h')

        # Reset for next fold
        if CHECKPOINT_PATH.exists():
            os.remove(CHECKPOINT_PATH)  # clean up — fold is done
        ep_budget = N_EPOCHS_THIS_SESSION

    # ── Save session state (after every fold or session stop) ──────────
    state = {
        'completed_folds': completed_folds,
        'current_fold': fold_id if session_done else fold_id + 1,
        'current_fold_epoch': epoch + 1 if session_done else 0,
        'fold_aucs': {str(k): v for k, v in fold_results.items()},
        'backbone': BACKBONE, 'in_channels': IN_CHANNELS,
        'n_classes': N_CLASSES, 'n_folds': N_FOLDS,
    }
    with open(SESSION_STATE_PATH, 'w') as f:
        json.dump(state, f, indent=2)

    # Clean up GPU
    del model, optimizer, scheduler, scaler, train_ds, val_ds
    torch.cuda.empty_cache()
    gc.collect()

total_h = (time.time() - session_start) / 3600
print(f'\n{"="*65}')
print(f'SESSION COMPLETE — {total_h:.1f}h')
print(f'{"="*65}')
if session_done:
    print(f'Stopped mid-way. Commit this notebook, then resume.')
    print(f'  Next session: set PREV_CHECKPOINT_DIR to this output.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  LEARNING CURVES (same plots as NB03)  ██
# ══════════════════════════════════════════════════════════════════════════
log = pd.read_csv(METRICS_LOG_PATH)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for fold_id in log['fold'].unique():
    flog = log[log['fold'] == fold_id]
    axes[0].plot(flog['epoch'], flog['train_loss'], label=f'F{fold_id} train', alpha=0.7)
    axes[0].plot(flog['epoch'], flog['val_loss'], '--', label=f'F{fold_id} val', alpha=0.7)
axes[0].set(title='Loss', xlabel='Epoch', ylabel='BCE Loss')
axes[0].legend(fontsize=7)

for fold_id in log['fold'].unique():
    flog = log[log['fold'] == fold_id]
    axes[1].plot(flog['epoch'], flog['auc_any'], label=f'F{fold_id} any')
    axes[1].plot(flog['epoch'], flog['auc_mean'], '--', label=f'F{fold_id} mean', alpha=0.7)
axes[1].set(title='Validation AUC', xlabel='Epoch', ylabel='AUC')
axes[1].set_ylim([0.5, 1.0])
axes[1].legend(fontsize=7)

for fold_id in log['fold'].unique():
    flog = log[log['fold'] == fold_id]
    axes[2].plot(flog['epoch'], flog['sensitivity'], label=f'F{fold_id} Sens')
    axes[2].plot(flog['epoch'], flog['specificity'], '--', label=f'F{fold_id} Spec', alpha=0.7)
axes[2].set(title='Sensitivity / Specificity', xlabel='Epoch')
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.savefig('/kaggle/working/learning_curves.png', bbox_inches='tight', dpi=150)
plt.show()

print(log[['fold', 'epoch', 'train_loss', 'val_loss',
           'auc_any', 'auc_mean', 'sensitivity', 'specificity', 'f1']].to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  HEALTH CHECK  ██
# ══════════════════════════════════════════════════════════════════════════
errors = []

# Check session state
if not SESSION_STATE_PATH.exists():
    errors.append('session_state.json missing')

# Check metrics log
if not METRICS_LOG_PATH.exists():
    errors.append('training_metrics.csv missing')
else:
    log_hc = pd.read_csv(METRICS_LOG_PATH)
    if len(log_hc) == 0:
        errors.append('Metrics log is empty')
    last_auc = log_hc['auc_any'].iloc[-1]
    if last_auc < 0.55:
        errors.append(f'Final AUC={last_auc:.4f} suspiciously low')
    nan_total = log_hc.get('nan_batches', pd.Series([0])).sum()
    if nan_total > 20:
        errors.append(f'{nan_total} NaN batches total')

# Check fold models
for k in completed_folds:
    p = BEST_MODEL_DIR / f'best_model_fold{k}.pth'
    if not p.exists():
        errors.append(f'best_model_fold{k}.pth missing')

# Check plots
if not os.path.exists('/kaggle/working/learning_curves.png'):
    errors.append('learning_curves.png missing')

health = {
    'notebook': '01_training',
    'status': 'PASS' if not errors else 'FAIL',
    'errors': errors,
    'completed_folds': completed_folds,
    'fold_aucs': {str(k): round(v, 5) for k, v in fold_results.items()},
    'total_epochs_trained': len(log_hc) if METRICS_LOG_PATH.exists() else 0,
    'session_hours': round((time.time()-session_start)/3600, 2),
}
with open('/kaggle/working/health_check_nb01.json', 'w') as f:
    json.dump(health, f, indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:')
    for e in errors: print(f'   • {e}')
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   Completed folds : {completed_folds}')
    for k, v in fold_results.items():
        print(f'   Fold {k} best AUC: {v:.5f}')
    if len(fold_results) > 1:
        vals = list(fold_results.values())
        print(f'   Mean ± Std      : {np.mean(vals):.5f} ± {np.std(vals):.5f}')

print(f'\n   Session time    : {health["session_hours"]:.1f}h')

# Disk usage
total_mb = 0
print(f'\n   Saved files:')
for f in sorted(Path('/kaggle/working').glob('*')):
    sz = f.stat().st_size / 1e6
    total_mb += sz
    print(f'     {f.name:40s} {sz:>8.1f} MB')
print(f'     {"TOTAL":40s} {total_mb:>8.1f} MB')

if max(completed_folds, default=-1) < N_FOLDS - 1 or session_done:
    print(f'\n   NEXT: Commit → new session → set PREV_CHECKPOINT_DIR')
else:
    print(f'\n   All {N_FOLDS} folds complete!')
    print(f'   NEXT: run 02_ablation.ipynb')